In [1]:
from tkinter import Y
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import sys

print("Python version:", sys.version)
print("NumPy version:", np.__version__)

dataDirectory = '/home/dave/SCRAM/data/'
sourceDirectory = '/home/dave/SCRAM/delfic_output/'
outputShape = (512, 512)

# find the names of each dataset. There are 3 files for each dataset *_X.npy, *_Yd.npy, *_Yr.npy
def getDatasetNames():
    import os
    files = os.listdir(dataDirectory)
    datasets = set()
    for f in files:
        if f.endswith('_X.npy'):
            datasets.add(f[:-6])

    return datasets

# load a dataset
def loadDataset(datasetName):
  X = np.load(dataDirectory + datasetName + '_X.npy')
  Yd = np.load(dataDirectory + datasetName + '_Yd.npy')
  Yr = np.load(dataDirectory + datasetName + '_Yr.npy')
  try:
    Y = np.load(sourceDirectory + datasetName + f'/{datasetName}.pkl', allow_pickle=True)
  except:
    Y = np.zeros((X.shape[2], outputShape[0], outputShape[1]))
    print(f'No source data for {datasetName}')
  return X, Yd, Yr, Y

def getBatch(datasetNames, shuffle=True):
    batchsize = len(datasetNames)
    _X, _Yd, _Yr, _Y = loadDataset(datasetNames[0])
    sampleSize = _X.shape[0]
    nDetectors = _X.shape[1]
    nTimeSteps = _X.shape[2]

    nFeatures = _X.shape[3]

    X = np.zeros((batchsize * sampleSize, nDetectors, nTimeSteps, nFeatures)) # (X, Y, dR, Dose, dDose)
    Yd = np.zeros((batchsize * sampleSize, nDetectors, nTimeSteps)) # true dose
    Yr = np.zeros((batchsize * sampleSize, nDetectors, 2)) # true X, Y position
    Y  = np.zeros((batchsize * sampleSize, nTimeSteps, outputShape[0], outputShape[1])) # true image

    X[:sampleSize] = _X
    Yd[:sampleSize] = _Yd
    Yr[:sampleSize] = _Yr
    Y[:sampleSize] = _Y

    for i in range(1, batchsize):
      _X, _Yd, _Yr, _Y = loadDataset(datasetNames[i])
      X[i*sampleSize:(i+1)*sampleSize] = _X
      Yd[i*sampleSize:(i+1)*sampleSize] = _Yd
      Yr[i*sampleSize:(i+1)*sampleSize] = _Yr
      Y[i*sampleSize:(i+1)*sampleSize] = _Y

    if shuffle:
      indices = np.random.permutation(X.shape[0])
      X = X[indices]
      Yd = Yd[indices]
      Yr = Yr[indices]
      Y = Y[indices]

    #reshape X from (batchsize, nDetectors, nTimeSteps, nFeatures) to (batchsize, nTimeSteps, nDetectors, nFeatures)
    X = np.swapaxes(X, 1, 2)

    #reshape Yd from (batchsize, nDetectors, nTimeSteps) to (batchsize, nTimeSteps, nDetectors)
    Yd = np.swapaxes(Yd, 1, 2)

    # Convert numpy arrays to torch tensors
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X = torch.tensor(X, dtype=torch.float32).to(device)
    Yd = torch.tensor(Yd, dtype=torch.float32).to(device)
    Yr = torch.tensor(Yr, dtype=torch.float32).to(device)
    Y = torch.tensor(Y, dtype=torch.float32).to(device)

    return X, Yd, Yr, Y


datasetNames = getDatasetNames()

#get 5 random datasets
datasets = np.random.choice(list(datasetNames), 5, replace=False)

X, Yd, Yr, Y = getBatch(datasets)
print(X.shape)
print(Yd.shape)
print(Yr.shape)
print(Y.shape)


Python version: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]
NumPy version: 1.26.4
torch.Size([100, 93, 240, 5])
torch.Size([100, 93, 240])
torch.Size([100, 240, 2])
torch.Size([100, 93, 512, 512])


In [2]:
# build a GRU model that takes in the X, Y, dR, Dose, dDose for each detector and then reconstructs the field at each time step
# at each timestep, the output is a 2d array of radiation exposure rate. The field is expected to decay over time. 
# the output image is 512x512

class GRUReconstructionModel(nn.Module):
    def __init__(self, nDetectors, nFeatures, hidden_dim, output_height, output_width, num_layers=1):
        super(GRUReconstructionModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.nDetectors = nDetectors
        self.nFeatures = nFeatures
        self.output_height = output_height
        self.output_width = output_width
        self.num_layers = num_layers
        
        # Define GRU layer
        input_size = nDetectors * nFeatures
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        
        # Fully connected layer to map GRU output to image size
        self.fc = nn.Linear(hidden_dim, output_height * output_width)
        
    def forward(self, x):
        batch_size = x.size(0)  # Get batch size
        
        # Reshape input from (batch_size, nTimeSteps, nDetectors, nFeatures) to (batch_size, nTimeSteps, nDetectors * nFeatures)
        x = x.contiguous().view(batch_size, x.size(1), -1)
        
        # Initialize hidden state for GRU
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        
        # Pass data through GRU
        gru_out, _ = self.gru(x, h0)  # gru_out: [batch_size, timesteps, hidden_dim]
        
        # Reconstruct the image for each timestep
        images = self.fc(gru_out)  # [batch_size, timesteps, output_height * output_width]
        images = images.view(batch_size, x.size(1), self.output_height, self.output_width)  # [batch_size, timesteps, H, W]
        
        return images
    
nTimeSteps = X.shape[1]
nDetectors = X.shape[2]
nFeatures = X.shape[3]

model = GRUReconstructionModel(nDetectors=nDetectors, nFeatures=nFeatures, 
                               hidden_dim=128, output_height=512, 
                               output_width=512, num_layers=2)

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Forward pass to confirm the model works
output = model(X)
print(output.shape)  # [batch_size, timesteps, H, W]

OutOfMemoryError: CUDA out of memory. Tried to allocate 9.08 GiB. GPU 0 has a total capacity of 11.73 GiB of which 803.56 MiB is free. Including non-PyTorch memory, this process has 9.54 GiB memory in use. Of the allocated memory 9.37 GiB is allocated by PyTorch, and 47.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [26]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
n_epochs = 5

for epoch in range(n_epochs):
  model.train()
  
  for i in range(10):  # 10 batches
    #get 5 random datasets
    datasets = np.random.choice(list(datasetNames), 5, replace=False)
    X, Yd, Yr = getBatch(datasets)
    
    optimizer.zero_grad()
    output = model(X)
    
    loss = criterion(output, Yd)
    loss.backward()
    optimizer.step()
    
    print(f'Epoch {epoch+1}/{n_epochs}, Batch {i+1}/10, Loss: {loss.item():.4f}')

# Save model
torch.save(model.state_dict(), 'gru_reconstruction_model.pth')
print('Model saved')

/home/dave/miniconda3/envs/torch/lib/python3.10/site-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([100, 93, 240])) that is different to the input size (torch.Size([100, 93, 512, 512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (512) must match the size of tensor b (240) at non-singleton dimension 3